In [15]:
import pickle
from dgllife.model.model_zoo import WeavePredictor
import dgl
import dgl.nn as nn
import dgl.function as fn
import torch.nn as tnn
import torch
import torch.optim
import torch.nn.functional as F
from torch.utils.data import random_split
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from pprint import pprint
from sklearn.preprocessing import MinMaxScaler

In [18]:
with open('graph_large_ndatas.pickle', 'rb') as handle:
    ndatas = pickle.load(handle)

with open('graph_large_edatas.pickle', 'rb') as handle:
    edatas = pickle.load(handle)
    
with open('graphs_large.pickle', 'rb') as handle:
    graphs = pickle.load(handle)

In [14]:
from pprint import pprint
pprint(ndatas['8'])
pprint(edatas['8'])

[[1,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  0,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  0.15360070686260013,
  1.012,
  ['CCCH', 'CHHH']],
 [0,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  4,
  'H',
  1,
  25,
  1.01,
  2.2,
  1,
  0.10970328319093947,
  0.95,
  ['CHHH', 'C']],
 [0,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  5,
  'H',
  1,
  25,
  1.01,
  2.2,
  1,
  0.10971210858528169,
  0.95,
  ['CHHH', 'C']],
 [0,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  6,
  'H',
  1,
  25,
  1.01,
  2.2,
  1,
  0.1098724273916647,
  0.949,
  ['CHHH', 'C']],
 [1,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  2,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  0.15361526820664587,
  1.012,
  ['CCCH', 'CHHH']],
 [3,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  1,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  0.1535668742302571,
  1.012,
  ['CHHH', 'CCCH']],
 [1,
  'C',
  6,
  70,
  12.0,
  2.55,
  4,
  7,
  'H',
  1,
  25,
  1.01,
  2.2,
  1,
  0.11008143546352354,
  0.936,
  ['CCCH', 'C']],
 [8,
  'H',
  1,
  25,
  1.01,
  

In [19]:
def OHE_and_normalize(mol: str):
    n = []
    for i in range(graphs[mol].num_nodes()):
        for x in ndatas[mol]:
            if (x[0] == i):
                n.append(x[1:7]+[x[-1][0]])
                break
            elif (x[7] == i):
                n.append(x[8:14]+[x[-1][1]])
                break
    assert len(n) == graphs[mol].num_nodes()
    
    e = []
    for i, j in zip(graphs[mol].edges()[0].tolist(), graphs[mol].edges()[1].tolist()):
        for x in ndatas[mol]:
            if ((i, j) == (x[0], x[7])) or ((i, j) == (x[7], x[0])):
                e.append(x[-3:-1])
                break
        for y in edatas[mol].keys():
            if ((str(i), str(j)) == y) or ((str(j), str(i)) == y):
                e[-1].append(edatas[mol][y])
    assert len(e) == graphs[mol].num_edges()
            
    #n_df = pd.DataFrame(n)
    #encoded_column_1 = pd.get_dummies(n_df[0])
    #encoded_column_2 = pd.get_dummies(n_df[6], prefix="nei")
    #n_df = n_df.join(encoded_column_1)
    #n_df = n_df.join(encoded_column_2)
    #print(n_df)
    #df.drop(col, axis=1, inplace=True)
    # try:
    #df = df.join(encoded_column)
    # except ValueError as verr:
    #     if "overlap" in verr.__str__():
    #         df = df.merge(encoded_column, left_on="BR", right_on="BR")
    return n, e

In [20]:
sorted_keys = sorted([int(x) for x in ndatas.keys()])
n_all = []
e_all = []
for x in sorted_keys:
        x_n, x_e = OHE_and_normalize(str(x))
        n_all = n_all+x_n
        e_all = e_all+x_e


In [7]:
n_df = pd.DataFrame(n_all, columns=["element", "atomic_number", "radius", "mass", "electronegativity", "hybridisation", "nei"])
e_df = pd.DataFrame(e_all)
ele_encoded = pd.get_dummies(n_df["element"], prefix="ele_")
nei_encoded = pd.get_dummies(n_df["nei"], prefix="nei")
n_df.drop("element", axis=1, inplace=True)
n_df.drop("nei", axis=1, inplace=True)
n_df = n_df.join(ele_encoded)
n_df = n_df.join(nei_encoded)
norm = MinMaxScaler().fit(n_df)
norm_n_df = norm.transform(n_df)

In [8]:
comb_graph = dgl.batch([graphs[str(x)] for x in sorted_keys])
#nx.draw(dgl.to_networkx(graphs['21']), with_labels=True)

In [9]:
class GraphConv(tnn.Module):
    def __init__(self, in_feats, hid_feats, out_feats):
        super().__init__()
        self.conv1 = nn.SAGEConv(in_feats=in_feats, out_feats=hid_feats, bias=True, aggregator_type='mean')
        self.conv3 = nn.SAGEConv(in_feats=hid_feats, out_feats=out_feats, bias=True, aggregator_type='mean')
        
    def message_passing(self, g):

        g.update_all(fn.copy_e('e', 'm_e'), fn.sum('m_e', 'h_e'))
        g.update_all(fn.copy_u('h', 'm_n'), fn.sum('m_n', 'h_n'))
        g.ndata['h'] = torch.cat((g.ndata.pop('h_n'), g.ndata.pop('h_e')), 1)
        g.update_all(fn.copy_e('e', 'm_e'), fn.sum('m_e', 'h_e'))
        g.update_all(fn.copy_u('h', 'm_n'), fn.sum('m_n', 'h_n'))
        g.ndata['h'] = torch.cat((g.ndata.pop('h_n'), g.ndata.pop('h_e')), 1)
    
    def forward(self, graph, inputs):
        self.message_passing(graph)
        
        h = self.conv1(graph, inputs)
        h = F.relu(h)
        h = self.conv3(graph, h)
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.apply_edges(fn.u_dot_v('h', 'h', 'score'))
            return graph.edata['score']
        

In [10]:
e_star = e_df.drop(2, axis=1)
norm_e = MinMaxScaler().fit(e_star)
norm_e_star = norm_e.transform(e_star)

In [11]:
comb_graph.ndata['h'] = torch.from_numpy(norm_n_df.astype('float32'))
comb_graph.edata['e'] = torch.from_numpy(norm_e_star.astype('float32'))
comb_graph.edata['score'] = torch.from_numpy(e_df[2].to_numpy().astype('float32'))

In [22]:
train_sp, val_sp, test_sp = random_split(norm_e_star, [0.5, 0.25, 0.25])
train_bin_full, train_bin, val_bin, test_bin = (np.zeros(len(comb_graph.edata['e'])) for i in range(4))
for x in train_sp.indices:
    train_bin[x] = 1
for y in val_sp.indices:
    val_bin[y] = 1
for z in test_sp.indices:
    test_bin[z] =1

for x in range(len(train_bin_full)):
    train_bin_full[x] = 1
print(train_bin_full)
print(train_bin)

[1. 1. 1. ... 1. 1. 1.]
[0. 1. 0. ... 0. 1. 1.]


In [23]:
comb_graph.edata['train_mask'] = torch.from_numpy(train_bin_full).bool()
comb_graph.edata['val_mask'] = torch.from_numpy(val_bin).bool()
comb_graph.edata['test_mask'] = torch.from_numpy(test_bin).bool()

In [24]:
node_features = comb_graph.ndata['h']
edge_label = comb_graph.edata['score']
train_mask = comb_graph.edata['train_mask']


In [25]:
self_loop_g = dgl.add_self_loop(comb_graph)

In [26]:
model = GraphConv(self_loop_g.ndata['h'].shape[1], 20, 10)
optimizer = torch.optim.Adam(model.parameters())


In [28]:
optimizer.param_groups[0]['lr'] = 0.01

for epoch in range(10000):
        pred = model(comb_graph, node_features)
        loss = abs(pred[train_mask].flatten() - edge_label[train_mask]).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(loss.item())

42317.3828125
45395.4375
42613.4375
43157.78125
44114.625
44018.8125
43267.140625
42486.421875
42561.48828125
43191.22265625
43311.27734375
42892.40234375
42422.12890625
42438.0625
42741.890625
42900.09765625
42818.875
42572.77734375
42360.515625
42431.36328125
42624.6796875
42663.41015625
42516.90625
42356.8203125
42381.03515625
42495.59375
42535.1015625
42465.45703125
42359.390625
42346.8671875
42427.640625
42456.68359375
42394.01171875
42329.4375
42356.20703125
42403.76171875
42396.4140625
42345.1328125
42326.14453125
42363.53125
42377.09375
42342.2421875
42321.6875
42345.1875
42357.0859375
42336.5078125
42319.28125
42334.43359375
42343.36328125
42325.84375
42318.265625
42330.84375
42331.66796875
42318.24609375
42318.7578125
42327.84765625
42321.7109375
42314.3125
42321.453125
42322.47265625
42313.9453125
42316.19140625
42319.91015625
42313.5078125
42313.328125
42316.36328125
42312.43359375
42312.0078125
42314.85546875
42311.38671875
42311.04296875
42312.83984375
42309.97265625
4231

KeyboardInterrupt: 

In [27]:
model.load_state_dict(torch.load('model_parms_42k.pt'))

<All keys matched successfully>

In [48]:
(edge_label[train_mask])

tensor([180040.1094, 298029.2500, 268233.4688,  ..., 185079.9219,
        198301.7969, 123835.5703])

In [29]:
abs(pred[train_mask] - edge_label[train_mask]).mean()

tensor(80652.1484, grad_fn=<MeanBackward0>)

In [37]:
torch.save(model.state_dict(), 'model_parms_42k.pt')

In [ ]:
model = GraphConv(comb_graph.ndata['h'].shape[1], 50, 20, 10)
optimizer = torch.optim.Adam(model.parameters())

model_weave = WeavePredictor(comb_graph.ndata['h'].shape[1], comb_graph.edata['e'].shape[1], 

In [42]:
print(norm_e_star)

[[0.37620873 0.20721817]
 [0.08499796 0.18207624]
 [0.0850565  0.18207624]
 ...
 [0.00174659 0.10583942]
 [0.08424243 0.17964315]
 [0.08451123 0.17680454]]
